In [1]:
# input
pdb_anno_file = "../pdb/data/entryId-seqNum-resi-metalResi-pdbId.tsv"
uni_anno_file = "../uniprot/data/entryId-seqNum-resi-metalResi.tsv"
# output
site_anno_file = "./data/entryId-seqNum-resi-metalResi.tsv"

In [2]:
import pandas as pd
df_pdb = pd.read_table(pdb_anno_file, header=None, names=["seq_id", "seq_num", "resi", "metal_resi", "pdb_id"]).drop(columns=["pdb_id"])
df_uni = pd.read_table(uni_anno_file, header=None, names=["seq_id", "seq_num", "resi", "metal_resi"])

In [3]:
def get_anno_records(df):
    records = []

    for _, row in df.iterrows():
        seq_id = row['seq_id']
        seq_nums = row['seq_num'].split(",")
        resis = row['resi'].split(",")
        metal_resis = row['metal_resi'].split(",")
        for idx, num in enumerate(seq_nums):
            records.append({
                "seq_id": seq_id,
                "seq_num": num,
                "resi": resis[idx],
                "metal_resi": metal_resis[idx]
            })
    return records

df_merged = pd.DataFrame(get_anno_records(df_pdb) + get_anno_records(df_uni))

In [4]:
len(df_merged)

30113667

In [5]:
df_merged = df_merged.drop_duplicates(subset=["seq_id", "seq_num"])
len(df_merged)

28373636

In [6]:
df_merged.head()

,seq_id,seq_num,resi,metal_resi
0,A0A011,137,L,MG
1,A0A011,138,A,MG
2,A0A011,140,F,MG
3,A0A022MQ12,75,H,ZN
4,A0A022MQ12,77,H,ZN


In [7]:
import tqdm

records = []
for (seq_id,), df_seq in tqdm.tqdm(df_merged.groupby(by=['seq_id'])):

    df = df_seq.sort_values(by=["seq_num"])
    records.append({
        "seq_id": seq_id,
        "seq_num": ",".join([str(r) for r in df["seq_num"]]),
        "resi": ",".join([str(r) for r in df["resi"]]),
        "metal_resi": ",".join([str(r) for r in df["metal_resi"]]),
    })

100%|██████████| 7507856/7507856 [20:36<00:00, 6070.97it/s]


In [8]:
pd.DataFrame(records).to_csv(site_anno_file, sep="\t", index=None, header=None)